In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, datediff, current_date
import json
import os
from pyspark.sql import functions as sf
from pyspark.sql.window import Window

ACCESS_KEY = os.environ.get("AWS_ACCESS_KEY_ID", "forge-commerce-user")
SECRET_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "forge-commerce-pass")
S3_ENDPOINT = os.environ.get("AWS_S3_ENDPOINT", "http://minio:9000")
PREFIX = "customers"
RAW_BUCKET = "raw"
CLEANED_BUCKET = "cleaned"
CURATED_BUCKET = "curated"
RAW_PATH = f"s3a://{RAW_BUCKET}/{PREFIX}/"
CLEANED_PATH = f"s3a://{CLEANED_BUCKET}/{PREFIX}/"
CURATED_PATH = f"s3a://{CURATED_BUCKET}/{PREFIX}/"

In [2]:
spark = (
        SparkSession.builder.appName("test_customers")
        .master(os.environ.get("SPARK_MASTER", "spark://spark-master:7077"))
        .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY)
        .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY)
        .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        # Delta Lake configurations
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
        .getOrCreate()
    )

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/20 02:08:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.format("delta").load(CLEANED_PATH)

df.orderBy("customer_id", sf.desc("created_at")).select("customer_id", "name", "full_address", "email", "created_at").show(10)

print("Total Rows: ", df.count())
print("Total Distinct Customer IDs: ", df.select("customer_id").distinct().count())

26/03/20 02:09:07 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/03/20 02:09:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+----------------+--------------------+--------------------+-------------------+
|customer_id|            name|        full_address|               email|         created_at|
+-----------+----------------+--------------------+--------------------+-------------------+
|          1|  Michele Wagner|148 Hurley Locks,...|burnsruth@example...|2026-03-18 22:18:45|
|          1|  Adam Rodriguez|79316 Troy Bypass...|jamesshaw@example...|2026-03-18 22:18:09|
|          1|   Melinda White|833 Gregory Mount...|alexandra92@examp...|2026-03-18 22:16:44|
|          1| Stephanie Garza|9301 Dana Flats A...|mejiamorgan@examp...|2026-03-18 21:06:31|
|          1| Daniel Campbell|027 Perry Mews, S...|debbie14@example.net|2026-03-18 21:06:24|
|          1| Gregory Russell|495 Mcmillan Isla...|burnettwayne@exam...|2026-03-18 21:06:22|
|          1|James Cunningham|27036 Holmes Exte...|hansenlisa@exampl...|2026-03-18 21:06:19|
|          1| Heather Herring|55227 Heather Isl...|mooremichaela@exa..

Total Rows:  3050


Total Distinct Customer IDs:  1000


In [4]:
spark.stop()